In [70]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/intermediate")
OUT_DIR = Path("../out")
PEGEL_PATH = DATA_DIR / "pegel_kirchentellinsfurt.csv"
WEATHER_HOURLY_PATH = DATA_DIR / "weather_data_hourly_2000_2026.csv"

def load_pegel(pegel_path=PEGEL_PATH):
    pegel = pd.read_csv(pegel_path, sep=";", usecols=["Messzeitpunkt", "Messwert"], dtype="string")
    pegel["timestamp"] = pd.to_datetime(pegel.pop("Messzeitpunkt"), format="%d.%m.%Y %H:%M", errors="coerce")
    pegel["water_level"] = pd.to_numeric(pegel.pop("Messwert").str.replace(",", ".", regex=False), errors="coerce")
    pegel = pegel.dropna(subset=["timestamp", "water_level"]).sort_values("timestamp")
    return pegel


def load_hourly_weather(weather_path=WEATHER_HOURLY_PATH):
    weather = pd.read_csv(weather_path, parse_dates=["timestamp"])
    weather["timestamp"] = weather["timestamp"].dt.floor("h")
    weather["station_id"] = weather["station_id"].astype(str).str.zfill(5)
    return weather

In [71]:
weather_data_hourly = load_hourly_weather()
pegel_data = load_pegel()




In [72]:
# Pivot hourly weather data wide so each station-variable is a feature column.
weather_data_hourly_pivoted = weather_data_hourly.pivot(index='timestamp', columns='station_id', values=['temperature_air_2m', 'precipitation_amount', 'humidity_relative', 'pressure_vapor']).reset_index()
weather_data_hourly_pivoted.columns = ['_'.join(map(str, col)) for col in weather_data_hourly_pivoted.columns]
weather_data_hourly_pivoted.rename(columns={'timestamp_': 'timestamp'}, inplace=True)

In [67]:
# See if hourly weather data is always reported exactly on the hour
repoting_times = weather_data_hourly['timestamp'].dt.minute.value_counts()
print(repoting_times)

timestamp
0    1026006
Name: count, dtype: int64


In [73]:
LAGS = 6

weather_data_hourly_pivoted = weather_data_hourly_pivoted.sort_values("timestamp")
pegel_data = pegel_data.sort_values("timestamp").reset_index(drop=True)

feature_frames = [pegel_data]

for h in range(1, LAGS + 1):
    target_time = pegel_data["timestamp"] - pd.Timedelta(hours=h)


    matched_weather = (pd.DataFrame({"target_time": target_time})
        .merge(weather_data_hourly_pivoted, left_on="target_time", right_on="timestamp", how="left")
        .drop(columns=["target_time", "timestamp"]))

    matched_weather.columns = [f"{col}_lag_{h}h" for col in matched_weather.columns]

    feature_frames.append(matched_weather)

    water_lag = pegel_data[["timestamp", "water_level"]].copy()
    water_lag["timestamp"] += pd.Timedelta(hours=h)
    water_lag = (pegel_data[["timestamp"]]
        .merge(water_lag, on="timestamp", how="left")
        .drop(columns="timestamp"))
    water_lag.columns = [f"water_level_lag_{h}h"]
    feature_frames.append(water_lag)

result = pd.concat(feature_frames, axis=1)
result.to_parquet("../data/final/pegel_weather_6_hourly_lags.parquet", index=False)

assert len(result) == len(pegel_data)
assert "_row_id" not in result.columns
assert all(f"water_level_lag_{h}h" in result.columns for h in range(1, LAGS + 1))

In [74]:
# Check how many observations have missing values in the features
missing_values = result.isnull().sum()
print(missing_values)

timestamp                               0
water_level                             0
temperature_air_2m_02074_lag_1h    551218
temperature_air_2m_03278_lag_1h    545343
temperature_air_2m_03402_lag_1h    538009
                                    ...  
pressure_vapor_03278_lag_6h        545343
pressure_vapor_03402_lag_6h        538009
pressure_vapor_04931_lag_6h        513560
pressure_vapor_06275_lag_6h        551141
water_level_lag_6h                     10
Length: 128, dtype: int64
